# Phase 3: Full B2 Flipbook Pipeline Cross-Verification on Arbitrary Coordinates

This notebook provides rigorous, independent cross-verification of the entire **SPHEREx Blink Reduction Pipeline** (`clean.py` -> `align.py` -> `coadd.py` -> `render.py`) on **arbitrary sky coordinates**:
* **Tested Sky Coordinates:** $\text{RA} = 270.08150^\circ, \text{Dec} = +5.61080^\circ$ (arbitrary field position, not an official target center)
* **Detector:** SPHEREx-D2 ($1.10 - 1.62\,\mu\text{m}$)
* **Key Checks:**
  1. Live or cached image discovery across active mission collections (`spherex_dr1`, `spherex_qr3`)
  2. Visit / epoch temporal clustering (`group_epochs`)
  3. Astrometric TAN reprojection onto shared North-up, East-left grid ($29\times29$ odd pixels)
  4. Optimal inverse-variance coaddition with automatic defect inpainting
  5. Uniform global contrast scaling and PNG / FITS frame generation
  6. Manifest schema compliance with B2 fields (`release`, `flag_counts`, `pixel_source`, `collections_checked`, `newest_obs_date`)

In [1]:
import json
from pathlib import Path
import numpy as np
from astropy.io import fits
from PIL import Image

from backend.app import archive, config
from backend.app.render import render_frames, target_grid, cutout_size_deg

print("All pipeline modules successfully imported.")

All pipeline modules successfully imported.


In [2]:
# Arbitrary Random Coordinates (Field star region offset from Barnard)
ra_rand = 270.08150
dec_rand = 5.61080
det_rand = 2
key_rand = "random_pos_270_5"
out_test_dir = Path("scratch/test_frames")

print(f"Target Coordinates: RA={ra_rand:.5f}°, Dec={dec_rand:+.5f}°, Detector=D{det_rand}")

Target Coordinates: RA=270.08150°, Dec=+5.61080°, Detector=D2


In [3]:
# Execute the Full B2 Pipeline
manifest = render_frames(
    ra=ra_rand,
    dec=dec_rand,
    key=key_rand,
    label="Arbitrary Field Star",
    position_source="Random Arbitrary Coordinate Input",
    detector=det_rand,
    mode="coadd",
    max_per_visit=8,
    fov_arcmin=3.0,
    out_root=out_test_dir,
    stretch="linear",
)

print(f"render_frames completed! Generated {len(manifest['frames'])} visit frames.")

  visit 0 2025-08-26: 8 of 18 exposures, 0.0% flagged, 0.0% no value, 1.3431 µm
  visit 1 2026-03-16: 8 of 18 exposures, 0.0% flagged, 0.0% no value, 1.3476 µm
  visit 2 2026-08-21: 8 of 20 exposures, 0.0% flagged, 0.0% no value, 1.5127 µm
wrote scratch/test_frames/random_pos_270_5/d2/manifest.json (3 frames, mode coadd)
render_frames completed! Generated 3 visit frames.


In [4]:
# 1. Verify Top-Level Manifest Specifications
assert manifest["format"] == "spherex-blink-frames/1", "Invalid manifest format!"
assert manifest["band"] == det_rand, "Band must match detector!"
assert manifest["band_range_um"] == [1.10, 1.62], "D2 band range must be [1.10, 1.62]!"
assert manifest["size_px"] == 29, "Grid size must be 29x29!"
assert manifest["orientation"] == "north up, east left", "Orientation must be north-up, east-left!"
assert manifest["collections_checked"] == list(config.COLLECTIONS), "Missing collections_checked!"
assert "UTC" in manifest["newest_obs_date"], "newest_obs_date must be formatted in UTC!"
assert manifest["scale"]["vmin"] < manifest["scale"]["vmax"], "vmin must be less than vmax!"

print("Top-level manifest metadata schema verification PASSED!.")

Top-level manifest metadata schema verification PASSED!.


In [5]:
# 2. Cross-Verify Per-Visit Frames and Physical Artifacts
frame_dir = out_test_dir / key_rand / f"d{det_rand}"
assert (frame_dir / "manifest.json").exists(), "manifest.json missing on disk!"

for f in manifest["frames"]:
    epoch_id = f["epoch_id"]
    print(f"\nVerifying Visit {epoch_id} ({f['date_iso']}):")
    
    # Verify PNG files
    png_path = frame_dir / f["png"]
    assert png_path.exists(), f"Missing {png_path}"
    im = Image.open(png_path)
    assert im.size == (29, 29), f"PNG size {im.size} != (29, 29)"
    assert im.mode == "RGBA", f"PNG mode {im.mode} != RGBA"
    
    flags_path = frame_dir / f["flags_png"]
    assert flags_path.exists(), f"Missing {flags_path}"
    
    # Verify FITS file
    fits_path = frame_dir / f["fits"]
    assert fits_path.exists(), f"Missing {fits_path}"
    with fits.open(fits_path) as hdul:
        assert set(h.name for h in hdul) == {"PRIMARY", "IMAGE", "SIGMA", "FLAGGED", "HAS_VALUE"}
        assert hdul["IMAGE"].data.shape == (29, 29)
        assert hdul["IMAGE"].header["DETECTOR"] == det_rand
        assert hdul["IMAGE"].header["BUNIT"] == "MJy/sr"
    
    # Verify Required B2 Fields
    assert f["release"] in ("DR1", "QR3", "DR1_DEEP", "QR3_DEEP"), f"Invalid release {f['release']}"
    assert f["pixel_source"] in ("irsa-cutout", "aws-s3"), f"Invalid pixel_source {f['pixel_source']}"
    assert isinstance(f["flag_counts"], dict), "flag_counts must be a dictionary"
    for flag_name in config.BAD_FLAG_NAMES:
        assert flag_name in f["flag_counts"], f"Missing {flag_name} in flag_counts!"
        
    print(f"  - Release: {f['release']}")
    print(f"  - Pixel Source: {f['pixel_source']}")
    print(f"  - Exposures added: {f['n_images']} of {f['n_images_in_epoch']}")
    print(f"  - Target Wavelength: {f['wl_target_um']} µm (BW: {f['bw_target_um']} µm)")
    print(f"  - Flagged pixels: {f['pct_flagged']}%")

print("\nAll physical FITS, PNG, and metadata cross-verifications PASSED!.")


Verifying Visit 0 (2025-08-26 04:37 UTC):
  - Release: DR1
  - Pixel Source: aws-s3
  - Exposures added: 8 of 18
  - Target Wavelength: 1.3431 µm (BW: 0.0324 µm)
  - Flagged pixels: 0.0%

Verifying Visit 1 (2026-03-15 04:43 UTC):
  - Release: DR1
  - Pixel Source: aws-s3
  - Exposures added: 8 of 18
  - Target Wavelength: 1.3476 µm (BW: 0.0327 µm)
  - Flagged pixels: 0.0%

Verifying Visit 2 (2026-08-21 15:52 UTC):
  - Release: QR3
  - Pixel Source: aws-s3
  - Exposures added: 8 of 20
  - Target Wavelength: 1.5127 µm (BW: 0.0369 µm)
  - Flagged pixels: 0.0%

All physical FITS, PNG, and metadata cross-verifications PASSED!.
